## Iteration Changelog
- Built an exploratory-first data cleaning workflow from raw schema inspection through feature engineering.
- Added explicit feature keep/remove rationale, missing value handling, and normalization decisions.
- Added deterministic export to `data/cleanedGambling.csv` for downstream notebooks.

## Raw Data Discovery

### Objective
Approach this dataset like a first-pass analyst:
1) inspect schema and quality,
2) perform broad EDA,
3) decide what to keep/remove,
4) engineer additional features,
5) address missingness,
6) normalize and export a model-ready dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.features.poker_hand_strength import build_stage_feature_payload, parse_cards

pd.set_option("display.max_columns", 200)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_PATH = ROOT / "data" / "gambling.csv"
OUT_PATH = ROOT / "data" / "cleanedGambling.csv"

df_raw = pd.read_csv(RAW_PATH)
print(f"Loaded {RAW_PATH}")
print(f"Rows: {len(df_raw):,} | Cols: {len(df_raw.columns)}")
df_raw.head(3)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# First-pass schema and missingness profile
schema = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=False),
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

print("Top columns by missingness:")
print(schema.head(15).to_string())

schema.head(20)

## Feature Keep/Remove Strategy

### Keep (core modeling context)
- hand identity/context: `hand_id`, `tourn_id`, `table`, `date`, `time`, `position`
- card + board columns: `cards`, `board_flop`, `board_turn`, `board_river`
- betting/pot columns: `bet_*`, `pot_*`, `stack`, `blinds`
- outcome columns for supervised labels: `result`, `balance`

### Drop/De-prioritize
- direct player identifiers (`name`) for generalization
- columns that are redundant with engineered features or mostly constant
- text-heavy action columns retained only for audit context, not as normalized predictors

In [ ]:
df = df_raw.copy()

# Normalize text fields used by feature builders and EDA
text_cols = ["result", "position", "cards", "board_flop", "board_turn", "board_river", "blinds"]
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.lower()

# Normalize numerics
numeric_cols = [
    "stack", "balance", "pot_pre", "pot_flop", "pot_turn", "pot_river",
    "bet_pre", "bet_flop", "bet_turn", "bet_river"
]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Labels
result_s = df.get("result", "").fillna("").astype(str)
df["won_flag"] = result_s.str.contains("took chips|won", regex=True).astype(int)
df["lost_flag"] = result_s.str.contains("lost|gave up", regex=True).astype(int)

# Missing-value strategy
# - numeric: fill with 0 where semantic default is "no value recorded yet"
# - text card/board tokens: keep as-is and parse safely with helper
for c in numeric_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0.0)

df[["won_flag", "lost_flag"]].mean().to_frame("rate")

In [ ]:
# Build stage-aware engineered features aligned with current model pipeline
stages = ["preflop", "flop", "turn", "river"]

stack_by_hand = (
    df.groupby("hand_id")["stack"]
    .apply(lambda s: [float(x) for x in s.fillna(0.0).tolist()])
    .to_dict()
)

def _safe(v):
    try:
        x = float(v)
        return 0.0 if np.isnan(x) else x
    except Exception:
        return 0.0

stage_frames = {}
for stage in stages:
    payloads = []
    for _, row in df.iterrows():
        hole = parse_cards(row.get("cards"))
        flop = parse_cards(row.get("board_flop"))
        turn = parse_cards(row.get("board_turn"))
        river = parse_cards(row.get("board_river"))

        board = []
        if stage in {"flop", "turn", "river"}:
            board += flop
        if stage in {"turn", "river"}:
            board += turn
        if stage == "river":
            board += river

        if stage == "preflop":
            total_bet = _safe(row.get("bet_pre"))
            current_pot = _safe(row.get("pot_pre"))
        elif stage == "flop":
            total_bet = _safe(row.get("bet_pre")) + _safe(row.get("bet_flop"))
            current_pot = _safe(row.get("pot_flop")) or _safe(row.get("pot_pre"))
        elif stage == "turn":
            total_bet = _safe(row.get("bet_pre")) + _safe(row.get("bet_flop")) + _safe(row.get("bet_turn"))
            current_pot = _safe(row.get("pot_turn")) or _safe(row.get("pot_flop"))
        else:
            total_bet = _safe(row.get("bet_pre")) + _safe(row.get("bet_flop")) + _safe(row.get("bet_turn")) + _safe(row.get("bet_river"))
            current_pot = _safe(row.get("pot_river")) or _safe(row.get("pot_turn"))

        bb = 2.0
        blind_text = str(row.get("blinds", "1/2"))
        if "/" in blind_text:
            try:
                bb = float(blind_text.split("/")[-1].strip())
            except Exception:
                bb = 2.0

        payload = build_stage_feature_payload(
            stage,
            hole,
            board,
            total_bet=total_bet,
            current_pot=current_pot,
            position=str(row.get("position", "")),
            hero_stack=_safe(row.get("stack")),
            table_stacks=stack_by_hand.get(row.get("hand_id"), []),
            big_blind=bb,
        )
        payloads.append(payload)

    stage_frames[stage] = pd.DataFrame(payloads).add_prefix(f"{stage}_")

feature_df = pd.concat([stage_frames[s] for s in stages], axis=1)
print("Engineered feature frame:", feature_df.shape)
feature_df.head(2)

In [ ]:
# Post-feature EDA snapshot (distribution shifts)
eda_cols = [
    "flop_hand_strength", "turn_hand_strength", "river_hand_strength",
    "flop_board_shared_strength_risk", "turn_board_shared_strength_risk", "river_board_shared_strength_risk",
    "preflop_hero_stack_bb", "flop_spr", "turn_spr", "river_spr"
]
eda_cols = [c for c in eda_cols if c in feature_df.columns]

feature_df[eda_cols].describe().T.head(12)

In [ ]:
# Assemble cleaned dataset
base_keep = [
    "hand_id", "tourn_id", "table", "date", "time", "table_size", "playing", "seat", "position",
    "cards", "board_flop", "board_turn", "board_river", "stack", "result", "balance", "won_flag", "lost_flag"
]
base_keep = [c for c in base_keep if c in df.columns]
cleaned = pd.concat([df[base_keep].reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)

# Normalize engineered numeric features (z-score, robust clipping)
engineered_cols = [c for c in cleaned.columns if c not in base_keep and pd.api.types.is_numeric_dtype(cleaned[c])]
for c in engineered_cols:
    s = pd.to_numeric(cleaned[c], errors="coerce").fillna(0.0)
    std = float(s.std(ddof=0))
    if std > 0:
        cleaned[c] = ((s - s.mean()) / std).clip(-6, 6)
    else:
        cleaned[c] = 0.0

print(f"Final shape: {cleaned.shape}")
cleaned.head(2)

In [ ]:
# Deterministic export + audit summary
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned = cleaned[sorted(cleaned.columns)]
cleaned.to_csv(OUT_PATH, index=False)

miss = cleaned.isna().sum().sort_values(ascending=False)
print(f"Saved -> {OUT_PATH}")
print(f"Rows: {len(cleaned):,} | Cols: {len(cleaned.columns)}")
print("Top missing counts:")
print(miss.head(10).to_string())
print("\nFirst 20 columns:")
print(cleaned.columns[:20].tolist())

## Iteration Changelog
- Refreshed preprocessing to align with current card-centric + stack-aware + board-risk feature pipeline.
- Added deterministic cleaned dataset export to `data/cleanedGambling.csv`.
- Added schema/quality checks for model-ready features used across stage inference.

## Preprocessing & Feature Engineering

### Dataset Overview
This notebook creates the canonical cleaned dataset used by downstream EDA, t-tests, PCA, and stage model diagnostics.

### Output
- Input: `data/gambling.csv`
- Final normalized output: `data/cleanedGambling.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from scripts.features.poker_hand_strength import build_stage_feature_payload, parse_cards

pd.set_option("display.max_columns", 200)

ROOT = Path.cwd()
RAW_PATH = ROOT / "data" / "gambling.csv"
OUT_PATH = ROOT / "data" / "cleanedGambling.csv"

print(f"Reading: {RAW_PATH}")
df = pd.read_csv(RAW_PATH)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

In [ ]:
# Normalize key categorical text fields
for col in ["result", "position", "action_pre", "action_flop", "action_turn", "action_river"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

# Ensure numeric betting/pot columns are numeric and non-null
num_cols = [
    "stack", "pot_pre", "pot_flop", "pot_turn", "pot_river",
    "bet_pre", "bet_flop", "bet_turn", "bet_river", "balance"
]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# Create binary outcome flags
result_series = df.get("result", "").astype(str)
df["won_flag"] = result_series.str.contains("took chips|won", regex=True).astype(int)
df["lost_flag"] = result_series.str.contains("lost|gave up", regex=True).astype(int)

df[["won_flag", "lost_flag"]].mean().to_frame("rate")

In [ ]:
# Build stage-aware model features (preflop/flop/turn/river) for each row
stages = ["preflop", "flop", "turn", "river"]

stack_by_hand = (
    df.groupby("hand_id")["stack"]
    .apply(lambda s: [float(x) for x in s.fillna(0.0).tolist()])
    .to_dict()
)

def _safe_float(x, default=0.0):
    try:
        v = float(x)
        if np.isnan(v):
            return default
        return v
    except Exception:
        return default

feature_frames = {}
for stage in stages:
    rows = []
    for _, row in df.iterrows():
        hole = parse_cards(row.get("cards"))
        flop = parse_cards(row.get("board_flop"))
        turn = parse_cards(row.get("board_turn"))
        river = parse_cards(row.get("board_river"))

        board = []
        if stage in {"flop", "turn", "river"}:
            board += flop
        if stage in {"turn", "river"}:
            board += turn
        if stage == "river":
            board += river

        if stage == "preflop":
            total_bet = _safe_float(row.get("bet_pre"))
            current_pot = _safe_float(row.get("pot_pre"))
        elif stage == "flop":
            total_bet = _safe_float(row.get("bet_pre")) + _safe_float(row.get("bet_flop"))
            current_pot = _safe_float(row.get("pot_flop")) or _safe_float(row.get("pot_pre"))
        elif stage == "turn":
            total_bet = _safe_float(row.get("bet_pre")) + _safe_float(row.get("bet_flop")) + _safe_float(row.get("bet_turn"))
            current_pot = _safe_float(row.get("pot_turn")) or _safe_float(row.get("pot_flop"))
        else:
            total_bet = _safe_float(row.get("bet_pre")) + _safe_float(row.get("bet_flop")) + _safe_float(row.get("bet_turn")) + _safe_float(row.get("bet_river"))
            current_pot = _safe_float(row.get("pot_river")) or _safe_float(row.get("pot_turn"))

        blinds = str(row.get("blinds", "1/2"))
        bb = 2.0
        if "/" in blinds:
            try:
                bb = float(blinds.split("/")[-1].strip())
            except Exception:
                bb = 2.0

        payload = build_stage_feature_payload(
            stage,
            hole,
            board,
            total_bet=total_bet,
            current_pot=current_pot,
            position=str(row.get("position", "")),
            hero_stack=_safe_float(row.get("stack")),
            table_stacks=stack_by_hand.get(row.get("hand_id"), []),
            big_blind=bb,
        )
        rows.append(payload)

    stage_df = pd.DataFrame(rows).add_prefix(f"{stage}_")
    feature_frames[stage] = stage_df

feature_df = pd.concat([feature_frames[s] for s in stages], axis=1)
print(feature_df.shape)
feature_df.head(2)

In [ ]:
# Merge engineered features into cleaned dataset
keep_base = [
    "hand_id", "tourn_id", "table", "date", "time", "table_size", "playing", "seat", "name",
    "position", "cards", "board_flop", "board_turn", "board_river", "stack", "result", "balance",
    "won_flag", "lost_flag"
]
keep_base = [c for c in keep_base if c in df.columns]

cleaned = pd.concat([df[keep_base].reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)

# Basic normalization for broad numeric comparability (z-score, clipped)
num_engineered = [c for c in cleaned.columns if c not in keep_base and cleaned[c].dtype != "object"]
for col in num_engineered:
    s = pd.to_numeric(cleaned[col], errors="coerce").fillna(0.0)
    std = s.std(ddof=0)
    if std > 0:
        cleaned[col] = ((s - s.mean()) / std).clip(-6, 6)
    else:
        cleaned[col] = 0.0

print(f"Cleaned rows: {len(cleaned):,} | columns: {len(cleaned.columns)}")
cleaned[num_engineered[:8]].describe().T.head(8)

In [ ]:
# Quality checks + deterministic export
missing = cleaned.isna().sum().sort_values(ascending=False)
print("Top missing columns:")
print(missing.head(10).to_string())

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
ordered_cols = sorted(cleaned.columns)
cleaned = cleaned[ordered_cols]
cleaned.to_csv(OUT_PATH, index=False)

print(f"Saved cleaned dataset -> {OUT_PATH}")
print(f"Rows: {len(cleaned):,}")
print(f"Cols: {len(cleaned.columns)}")
print("Sample columns:", cleaned.columns[:15].tolist())